# Reto de Vision: Clasificacion de Productos con CNN

## Notebook 2 - Arquitectura, compilacion y entrenamiento de la CNN

### Donde quedamos

En la sesion anterior entendimos, a mano y sin usar Keras, que hace una convolucion y que hace un max pooling. Despues cargamos `Fashion MNIST`, lo normalizamos (pixeles entre 0 y 1) y le dimos la forma 4D `(batch, 28, 28, 1)` que necesita una capa `Conv2D`.

### Objetivos de esta sesion

- Repasar rapido la preparacion de datos para tener `X_train`, `y_train`, `X_test`, `y_test` listos.
- Entender cada capa de una arquitectura CNN y para que sirve.
- Construir con Keras una CNN con 2 capas `Conv2D` + `MaxPooling2D`, `Flatten` y capas `Dense`, terminando en una salida de 10 neuronas con `softmax`.
- Compilar el modelo (optimizador, funcion de perdida, metrica) y entrenarlo por al menos 10 epocas.
- Guardar el modelo entrenado y el historial de entrenamiento, para poder analizarlos a fondo en la proxima sesion sin tener que volver a entrenar.

En esta sesion no vamos a analizar en detalle los resultados (eso es la Fase 3, la proxima sesion). Hoy el foco esta en construir y entrenar el modelo correctamente.

## 1. Repaso rapido: preparando los datos otra vez

Como este es un notebook nuevo, no tenemos en memoria las variables que creamos la sesion pasada. Vamos a repetir, de forma rapida y sin volver a explicar cada detalle, los pasos de la Fase 1:

1. Cargar `Fashion MNIST` desde Keras.
2. Normalizar los pixeles a un rango de 0 a 1.
3. Hacer reshape para agregar el canal de color y llegar a un tensor 4D.
4. Guardar la lista de nombres de categorias, la vamos a necesitar para interpretar resultados mas adelante.

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import json

(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

X_train = X_train / 255.0
X_test = X_test / 255.0

X_train = X_train.reshape(X_train.shape[0], 28, 28, 1)
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1)

class_names = [
    "Playera/Top", "Pantalon", "Sueter", "Vestido", "Abrigo",
    "Sandalia", "Camisa", "Tenis", "Bolso", "Botin"
]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
X_train: (60000, 28, 28, 1)
X_test: (10000, 28, 28, 1)


## 2. Anatomia de la arquitectura que vamos a construir

Vamos a armar la siguiente secuencia de capas. Antes de programarla, repasemos que hace cada una:


```text
                    Imagen de entrada
                         (28x28x1)
                              │
                              ▼
                  ┌─────────────────────┐
                  │ Conv2D              │
                  │ 32 filtros (3x3)    │
                  │ Activacion: ReLU    │
                  └─────────────────────┘
                              │
                              ▼
                  ┌─────────────────────┐
                  │ MaxPooling2D (2x2)  │
                  └─────────────────────┘
                              │
                              ▼
                Feature maps mas pequeños
                              │
                              ▼
                  ┌─────────────────────┐
                  │ Conv2D              │
                  │ 64 filtros (3x3)    │
                  │ Activacion: ReLU    │
                  └─────────────────────┘
                              │
                              ▼
                  ┌─────────────────────┐
                  │ MaxPooling2D (2x2)  │
                  └─────────────────────┘
                              │
                              ▼
                  ┌─────────────────────┐
                  │      Flatten        │
                  │ Convierte todos los │
                  │ feature maps en un  │
                  │ solo vector         │
                  └─────────────────────┘
                              │
                              ▼
                  ┌─────────────────────┐
                  │ Dense               │
                  │ Activacion: ReLU    │
                  └─────────────────────┘
                              │
                              ▼
                  ┌─────────────────────┐
                  │ Dense (10 neuronas) │
                  │ Activacion: Softmax │
                  └─────────────────────┘
                              │
                              ▼
                 Probabilidad de cada clase
```
Un patron muy comun en CNN es que, a medida que avanzamos en profundidad, el tamaño espacial (alto x ancho) se va reduciendo por el pooling, mientras que la cantidad de filtros (profundidad) va aumentando. La idea es que la red va cambiando "donde esta el patron" por "que tan complejo es el patron".

>**¿Y por qué usamos Flatten y una Dense layer después?**
>
>La CNN ya hizo el trabajo de extraer features de la imagen.
>
>Flatten  toma todos los valores de todos los feature maps y los coloca uno detrás de otro en un solo vector.  
Feature maps:  
```
Feature maps:

[ 1  2 ]    [ 5  6 ]
[ 3  4 ]    [ 7  8 ]

       ↓ Flatten

[ 1, 2, 3, 4, 5, 6, 7, 8 ]
```
>
>La capa Dense toma todos esos features y los combina para decidir qué clase representa mejor la imagen.

>**¿Por qué usamos Softmax al final del modelo? (y no el sigmoid)**
>
> Nuestro modelo realiza una tarea de **multiclass classification**: tenemos **10 categorías posibles** y cada imagen pertenece a **una sola categoría**.
>
> Por eso necesitamos que las 10 neuronas finales compitan entre sí para determinar cuál es la categoría más probable.
>
> La función de activación **Softmax** es adecuada para este tipo de problema porque transforma los valores producidos por las 10 neuronas en una distribución de probabilidades:
>
```text
Valores de salida
       ↓
    Softmax
       ↓
Probabilidades
       ↓
[0.02, 0.01, 0.85, 0.03, ...]

## 3. Construyendo el modelo

### Que vamos a programar

Vamos a crear un modelo `Sequential` con las siguientes capas, en este orden:

1. `Conv2D` con 32 filtros, kernel de tamaño (3,3), activacion `relu`, y `input_shape=(28, 28, 1)` (solo la primera capa necesita `input_shape`).
2. `MaxPooling2D` con tamaño de pool (2,2).
3. `Conv2D` con 64 filtros, kernel de tamaño (3,3), activacion `relu`.
4. `MaxPooling2D` con tamaño de pool (2,2).
5. `Flatten`.
6. `Dense` con 128 neuronas y activacion `relu`.
7. `Dense` con 10 neuronas (una por categoria) y activacion `softmax`.

In [2]:
model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Revisando la arquitectura con `summary()`

El metodo `model.summary()` nos muestra, capa por capa, la forma (`shape`) de la salida y la cantidad de parametros entrenables. Es una herramienta muy util para verificar que el modelo quedo como lo pensamos, antes de gastar tiempo entrenandolo.

Fijate en dos cosas al mirar el resultado:

- Como el `shape` espacial (las dos primeras dimensiones despues del batch) se va achicando capa a capa, por efecto de las convoluciones (sin padding) y del pooling.
- Como la cantidad de parametros de las capas `Dense` es mucho mayor que la de las capas `Conv2D`. Esto pasa porque en una capa `Conv2D` los mismos filtros se reutilizan (comparten pesos) en toda la imagen, mientras que en una capa `Dense` cada conexion tiene su propio peso.

In [3]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

## 4. Compilando el modelo

Antes de entrenar, tenemos que compilar el modelo indicando tres cosas:

- **Optimizador**: `adam`, el mismo que usamos la semana pasada con redes densas. Se encarga de ajustar los pesos de la red durante el entrenamiento.
- **Funcion de perdida (`loss`)**: `sparse_categorical_crossentropy`. Usamos la version "sparse" porque nuestras etiquetas (`y_train`) son numeros enteros (0 a 9) y no vectores one-hot. Si hubieramos convertido las etiquetas a one-hot, usariamos `categorical_crossentropy` en su lugar.
- **Metrica**: `accuracy`, para poder ver que porcentaje de imagenes clasifica bien el modelo durante el entrenamiento.

### Que vamos a programar

Llamar a `model.compile()` con esos tres argumentos.

In [4]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

## 5. Entrenando el modelo

Ya tenemos la arquitectura definida y compilada. Ahora toca entrenar, usando `model.fit()`.

Algunos parametros importantes:

- `epochs`: cuantas veces la red va a recorrer todo el set de entrenamiento completo. Vamos a usar al menos 10.
- `batch_size`: cuantas imagenes procesa el modelo antes de actualizar los pesos una vez. Un valor comun es 32 o 64.
- `validation_data`: le pasamos `(X_test, y_test)` para que, al final de cada epoca, el modelo tambien calcule loss y accuracy sobre datos que no vio durante el entrenamiento. Esto nos va a servir muchisimo en la proxima sesion para detectar sobreajuste (overfitting).

Guardamos el resultado de `model.fit()` en una variable llamada `history`: ahi queda registrado el loss y el accuracy de cada epoca, tanto en entrenamiento como en validacion.

Ten en cuenta que entrenar una CNN tarda mas que una red densa sobre datos tabulares. Con 10 epocas y este dataset, deberia tardar unos minutos dependiendo de tu maquina.

In [5]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
)

Epoch 1/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.8137 - loss: 0.5191 - val_accuracy: 0.8633 - val_loss: 0.3828
Epoch 2/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8764 - loss: 0.3413 - val_accuracy: 0.8866 - val_loss: 0.3190
Epoch 3/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8945 - loss: 0.2901 - val_accuracy: 0.8915 - val_loss: 0.2979
Epoch 4/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9036 - loss: 0.2594 - val_accuracy: 0.9013 - val_loss: 0.2723
Epoch 5/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9117 - loss: 0.2355 - val_accuracy: 0.9082 - val_loss: 0.2514
Epoch 6/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9204 - loss: 0.2115 - val_accuracy: 0.9109 - val_loss: 0.2491
Epoch 7/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9280 - loss: 0.1910 - val_accuracy: 0.9074 - val_loss: 0.2552
Epoch 8/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9355 - loss: 0.1734 - val_accuracy: 0.

## 6. Una primera mirada rapida a los resultados

Sin entrar todavia en el analisis a fondo (eso lo hacemos en la proxima sesion con graficas y matriz de confusion), veamos el accuracy final que quedo registrado en el historial de entrenamiento, tanto en el set de entrenamiento como en el de validacion.

In [6]:
accuracy_final_train = history.history["accuracy"][-1]
accuracy_final_val = history.history["val_accuracy"][-1]

print("Accuracy final en entrenamiento: {:.2%}".format(accuracy_final_train))
print("Accuracy final en validacion: {:.2%}".format(accuracy_final_val))

Accuracy final en entrenamiento: 94.90%
Accuracy final en validacion: 90.74%


Si el accuracy de entrenamiento es bastante mas alto que el de validacion, es una señal de que el modelo podria estar memorizando el set de entrenamiento en vez de generalizar bien (overfitting). Vamos a estudiar esto con mas detalle en la proxima sesion.

## 7. Guardando el modelo y el historial de entrenamiento

Entrenar la CNN toma tiempo, y no queremos tener que repetir el entrenamiento en la proxima sesion solo para poder analizar los resultados. Por eso vamos a guardar dos cosas en disco:

1. El modelo entrenado completo (arquitectura + pesos), usando `model.save()`.
2. El historial de entrenamiento (`history.history`, que es un diccionario con las listas de loss y accuracy por epoca), usando `json.dump()`.

En la proxima sesion vamos a cargar estos dos archivos directamente, sin volver a entrenar nada.

In [7]:
model.save("modelo_cnn_fashion.keras")

with open("historial_entrenamiento.json", "w") as archivo:
    json.dump(history.history, archivo)

print("Modelo e historial guardados correctamente.")

Modelo e historial guardados correctamente.


## Resumen y proximos pasos

En esta sesion:

- Repasamos rapido la preparacion de los datos de `Fashion MNIST`.
- Entendimos el rol de cada capa en una arquitectura CNN: `Conv2D` para detectar patrones, `MaxPooling2D` para reducir tamaño, `Flatten` para poder conectar con capas densas, y `Dense` para combinar toda la informacion y llegar a una prediccion.
- Construimos, compilamos y entrenamos nuestra primera CNN completa sobre imagenes reales.
- Guardamos el modelo y el historial de entrenamiento para poder retomarlos sin re-entrenar.

En la proxima sesion (Fase 3) vamos a cargar el modelo guardado, graficar la evolucion de loss y accuracy, revisar en que categorias se confunde mas el modelo (matriz de confusion) y conectar esos errores con el impacto de negocio para la empresa de e-commerce.

## Extra: la misma arquitectura en PyTorch

Todo lo que hicimos en este notebook lo hicimos con Keras, que es una libreria de alto nivel: `Sequential`, `model.compile()` y `model.fit()` esconden mucho trabajo repetitivo para que puedas iterar rapido. Esto es ideal para prototipar y para produccion.

`PyTorch` es la otra libreria mas usada en Deep Learning, y toma un enfoque distinto: es mas explicito y manual. Vos escribis la clase del modelo, el forward pass, y el loop de entrenamiento con tus propias manos. Esto significa mas lineas de codigo, pero tambien mas control: por eso es la libreria preferida en investigacion, donde muchas veces necesitas hacer cosas no convencionales (loops de entrenamiento custom, arquitecturas raras, perdidas combinadas) que un `.fit()` no te deja hacer facilmente.

```python
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 1. La arquitectura se define como una clase que hereda de nn.Module
class CNNFashion(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3)
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
        self.pool2 = nn.MaxPool2d(kernel_size=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 5 * 5, 128)  # 5x5 sale de aplicar conv+pool dos veces sobre 28x28
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        # A diferencia de Keras, aca escribimos nosotros mismos el camino que sigue el dato
        x = self.relu(self.conv1(x))
        x = self.pool1(x)
        x = self.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)  # Ojo: no aplicamos softmax aca, ver el comentario de abajo
        return x

model = CNNFashion()

# 2. Funcion de perdida y optimizador (el equivalente a compile() en Keras)
# CrossEntropyLoss ya aplica softmax internamente, por eso la ultima capa del modelo
# devuelve "logits" crudos en vez de probabilidades ya calculadas
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# 3. PyTorch espera tensores con forma (lote, canales, alto, ancho), al reves que Keras,
# que usa (lote, alto, ancho, canales). Por eso usamos permute() para reordenar los ejes
X_train_t = torch.tensor(X_train, dtype=torch.float32).permute(0, 3, 1, 2)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32).permute(0, 3, 1, 2)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# 4. Entrenamiento: en Keras esto era una sola linea (model.fit()).
# En PyTorch escribimos el loop nosotros mismos, epoca por epoca y batch por batch
for epoch in range(10):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()          # Reiniciamos los gradientes acumulados
        outputs = model(X_batch)       # Forward pass
        loss = loss_fn(outputs, y_batch)
        loss.backward()                # Backward pass: calcula los gradientes
        optimizer.step()               # Actualiza los pesos del modelo

    print(f"Epoca {epoch + 1}/10 - Loss: {loss.item():.4f}")
```

La logica de fondo (convolucion, pooling, dense, adam, crossentropy) es exactamente la misma que ya programaste con Keras. Lo que cambia es cuanto hace el framework por vos: en Keras el `.fit()` maneja el loop de entrenamiento, el batching y el calculo de metricas por ti; en PyTorch, todo eso lo escribis vos mismo. Esto ultimo toma mas tiempo de escribir, pero te da control total sobre cada paso, lo cual es muy valioso cuando estas investigando algo nuevo que no encaja en el molde estandar.